# Scraping a news website for headlines

This code scrapes information from Le Monde's website (https://www.lemonde.fr/en/) to create a csv with each article on the homepages' title, subhead, article URL, whether it's premium or not, byline, article type and image URL. 

## Importing the necessary packages and data

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Getting the homepage data and saving it in a variable called doc

In [2]:
response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser')

## Searching the webpage data for the elements I need

Based on using the inspect tool on the website, I found the element div.article.article--headlines is being used to display several articles on the page. But that doesn't look like it will work for everything as different articles display on the page different. Thus I started by at any element with the article class.

In [3]:
stories = doc.find_all(class_="article")
stories

[<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96465"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96465" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/environment/article/2026/06/22/politicians-and-presidential-candidates-remain-in-denial-about-the-challenges-of-adapting-to-global-warming_6754740_114.html"> <p class="article__title-label">Presidential candidates in denial about the challenges of adaptation to climate change</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Towels hung over eighth-floor windows to protect residents from the heat, in the eastern town of Bavilliers, on June 21, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/21/0/0/4252/2834/400/266/75/0/825bf37_upload-1

#### First attempt at exploring the data using Soma's example code from class

In [4]:
# Starting off without ANY rows
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        print("Headline not found!")
        continue
        
    try:
        # Find me a media__link OR a reel_link
        row['href'] = story.select_one('a')['href']
    except:
        print("Couldn't find a link")

    try:
        row['tag'] = story.select_one('.ewmvhB').text.strip()
    except:
        print("Couldn't find a tag!")

    try:
        row['tag'] = story.find(attrs={'data-testid': 'card-description'}).text.strip()
    except:
        print("Couldn't find a summary")

    print(row)
    # When we're done adding info to our row, we're going to add it into our list
    # of rows
    rows.append(row)

----
Couldn't find a tag!
Couldn't find a summary
{'title': 'Presidential candidates in denial about the challenges of adaptation to climate change', 'href': 'https://www.lemonde.fr/en/environment/article/2026/06/22/politicians-and-presidential-candidates-remain-in-denial-about-the-challenges-of-adapting-to-global-warming_6754740_114.html'}
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!


That did not work at all. So I will start exploring the data on my own.

#### Exploring the first 3 and last 3 stories myself just to get a sense of what elements are in it

In [5]:
for story in stories[:3]:
    print("----")
    print(story)

print(" ----- last 3 ----- ")

for story in stories[(len(stories)-3):]:
    print("----")
    print(story)

----
<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96465"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96465" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/environment/article/2026/06/22/politicians-and-presidential-candidates-remain-in-denial-about-the-challenges-of-adapting-to-global-warming_6754740_114.html"> <p class="article__title-label">Presidential candidates in denial about the challenges of adaptation to climate change</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Towels hung over eighth-floor windows to protect residents from the heat, in the eastern town of Bavilliers, on June 21, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/21/0/0/4252/2834/400/266/75/0/825bf37_uplo

## Building a scraper to capture the information I need

We are looking for...<br>
headline = article__title class <br>
subtitle = article__desc class (only exists for some headlines) <br>
article url = lmd-link-clickarea__link class <br>
image url = src attribute withing the image tag <br>
bylines = article__byline class (but most aren't shown on the homepage) another option I have seen is article__author-name <br>
whether it is premium or not = sr-only class <br>
article type = article__type class (doesn't exist for all stories) <br>

This code uses the above information in a for loop to build a data set with the information we need. I am printing the information for each step to make sure it is working as planned.

In [6]:
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        # This gets the headlines generally but will need to be cleaned up using pandas
        row["headline"] = (story.select_one(".article__title").get_text())
        print(row["headline"])
    except:
        print("Headline not found!")
        continue
        
    try:
        # finding the subtitles
        row["subtitle"] = (story.select_one(".article__desc").get_text())
        print(row["subtitle"])
    except:
        print("Couldn't find a subtitle")

    # The .lmd-link-clickarea__link is not getting the article urls for everything unfortunately
    # so I have added one more attempt to catch stragglers before instructing the loop to give up
    try:
        row["article_url"] = story.select_one(".lmd-link-clickarea__link")["href"]
        print(row["article_url"])
    except:
        try:
            row["article_url"] = story.select_one("a")["href"]
            print(row["article_url"])
        except:
            print("Couldn't find an article url")

    try:
        row["image_url"] = story.select_one("img")["src"]
        print(row["image_url"])
    except:
        print("Couldn't find an image_url")
    
    # Bylines appear under different names and many of the articles do not have bylines at all on the homepage
    try:
        row["byline"] = (story.select_one(".article__byline").get_text())
        print(f"byline {row["byline"]}")
    except:
        try:
            row["byline"] = (story.select_one(".article__author-name").get_text())
            print(f"author name {row["byline"]}")
        except:
            print("Couldn't find a byline")

    try:
        if story.select_one(".sr-only") != None:
            row["paywall_status"] = "subscriber only"
            print(row["paywall_status"])
    except:
        print("Couldn't find subscriber status")

    try:
        row["article_type"] = (story.select_one(".article__type").get_text())
        print(row["article_type"])
    except:
        print("Couldn't find article type")
        
    # # When we're done adding info to our row, we're going to add it into our list    
    print(row)
    rows.append(row)

----
 Presidential candidates in denial about the challenges of adaptation to climate change  
As a severe heat wave is disrupting life in France, adapting to climate change remains a blind spot ahead of the 2027 presidential election.
https://www.lemonde.fr/en/environment/article/2026/06/22/politicians-and-presidential-candidates-remain-in-denial-about-the-challenges-of-adapting-to-global-warming_6754740_114.html
https://img.lemde.fr/2026/06/21/0/0/4252/2834/400/266/75/0/825bf37_upload-1-elb9m3h9vwpy-canicule-bavillers007-1.jpg
Couldn't find a byline
subscriber only
Couldn't find article type
{'headline': ' Presidential candidates in denial about the challenges of adaptation to climate change  ', 'subtitle': 'As a severe heat wave is disrupting life in France, adapting to climate change remains a blind spot ahead of the 2027 presidential election.', 'article_url': 'https://www.lemonde.fr/en/environment/article/2026/06/22/politicians-and-presidential-candidates-remain-in-denial-about-t

## Exporting the results

Saving it as a data frame and having a quick glance

In [7]:
df = pd.DataFrame(rows)
df

,headline,subtitle,article_url,image_url,paywall_status,article_type,byline
0,Presidential candidates in denial about the c...,As a severe heat wave is disrupting life in Fr...,https://www.lemonde.fr/en/environment/article/...,https://img.lemde.fr/2026/06/21/0/0/4252/2834/...,subscriber only,NaN,NaN
1,UK Prime Minister Keir Starmer announces resig...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/22/1/0/5500/3666/...,NaN,NaN,NaN
2,"Branko Milanovic, economist: 'There will be no...",NaN,https://www.lemonde.fr/en/opinion/article/2026...,https://img.lemde.fr/2026/06/10/1260/0/3808/25...,subscriber only,NaN,NaN
3,Colombia swings to far right as Abelardo de la...,"An outsider before the vote, the far-right law...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/22/60/0/3600/2400...,subscriber only,NaN,NaN
4,How Ukraine is orchestrating a logistical sque...,"Using a new generation of drones, Kyiv is wagi...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/21/0/87/5326/3551...,subscriber only,NaN,NaN
5,"Ten years after Brexit, Britons want closer ti...",According to a survey by the European Council ...,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/20/1/0/7785/5190/...,subscriber only,NaN,NaN
6,France celebrates the Fête de la Musique despi...,Despite some cancellations and with a large po...,https://www.lemonde.fr/en/environment/article/...,https://img.lemde.fr/2026/06/21/0/0/6961/4641/...,subscriber only,Feature,NaN
7,Free parties in France come under increasing s...,NaN,https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/06/19/27/0/640/426/1...,subscriber only,NaN,NaN
8,"What are prediction markets, the new form of s...",NaN,https://www.lemonde.fr/en/les-decodeurs/articl...,https://img.lemde.fr/2026/06/19/0/0/8256/5504/...,subscriber only,NaN,NaN
9,"In Paris, thousands march against far right in...",NaN,https://www.lemonde.fr/en/politics/article/202...,https://img.lemde.fr/2026/06/21/502/0/1890/126...,subscriber only,NaN,NaN


Exporting the data as a csv

In [8]:
df.to_csv("le_monde_daily_news.csv")

## Making the CSV auto-updating

I have created a github action to autoupdate this code, using Jonathan Soma's tutorial
https://www.youtube.com/watch?v=QNKxzkNpsko.

For future reference:
Step 1: Make your working files into a github repository
Step 2: Create a github action called update.yml using Soma's code 
Step 3: Make sure that you use a Cron code for the frequency of update that you want and that the file referenced is the correct one. Try to run the action and trouble shoot any error by checking at which point the code stopped being able to run.
